# Stage-2C five-stream residual Transformer

Loads both complete frozen DINOv3-B branches from the latest Stage-2C checkpoint, initializes the Transformer from scratch, and trains only a bounded residual over the bottle-DINO ranking. Inputs are the persisted 45k and Manual-211 crops.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_stage2c_residual')
OUTPUT = Path('/kaggle/working/stage2c_residual_transformer')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def input_root(slug):
    matches = sorted({path.resolve() for pattern in (slug, f'*/{slug}', f'*/*/{slug}') for path in INPUT.glob(pattern) if path.is_dir()})
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one {slug}; found: {matches}')
    return matches[0]

def unique_file(root, filename):
    matches = list(root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {filename} under {root}; found: {matches}')
    return matches[0]

print('SETUP | resolving Stage-2C residual inputs...', flush=True)
code = input_root('viscaner-stage2c-residual-code')
stage2c_root = input_root('viscaner-stage2c-checkpoint')
easyocr_weights = input_root('viscaner-easyocr-ru-en-weights')
hard = input_root('viscaner-fusion-hardset-v1')
labels = input_root('viscaner-dinov3-data')
bottles = input_root('viscaner-bottle-classifier-data')
manual = input_root('viscaner-manual-211')
stage2c_checkpoint = unique_file(stage2c_root, 'manual_stage2c_best.pt')

shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
for archive in PROJECT.glob('*.zip'):
    destination = PROJECT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
checksums = json.loads((PROJECT/'code_sha256.json').read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {name}')
stage2c_checksums = json.loads((stage2c_root/'stage2c_sha256.json').read_text())
actual_stage2c = hashlib.sha256(stage2c_checkpoint.read_bytes()).hexdigest()
if actual_stage2c != stage2c_checksums['manual_stage2c_best.pt']:
    raise RuntimeError('Stage-2C checkpoint checksum mismatch')
ocr_checksums = json.loads((easyocr_weights/'weights_sha256.json').read_text())
for name, expected in ocr_checksums.items():
    actual = hashlib.sha256((easyocr_weights/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'OCR weight checksum mismatch: {name}')
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
hard = materialize_dataset_dirs(hard, ('label_crops', 'bottle_crops'), cache)
labels = materialize_dataset_dirs(labels, ('refs', 'crops'), cache)
bottles = materialize_dataset_dirs(bottles, ('refs', 'crops'), cache)
manual = materialize_dataset_dirs(manual, ('label_crops', 'bottle_crops'), cache)
for required in (hard/'manifest.csv', labels/'refs', labels/'crops', bottles/'refs', bottles/'crops', manual/'manifest.csv', stage2c_checkpoint, easyocr_weights/'craft_mlt_25k.pth', easyocr_weights/'cyrillic_g2.pth'):
    if not required.exists():
        raise FileNotFoundError(required)
print(f'SETUP | verified {len(checksums)} code files and Stage-2C sha256={actual_stage2c}', flush=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'Unidecode>=1.3,<2', 'easyocr==1.7.2'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator before running this notebook')
print('SETUP | GPU:', torch.cuda.get_device_name(0), '| Stage-2C branches only', flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'five_stream_transformer.train',
    '--hard-manifest', str(hard/'manifest.csv'), '--hard-root', str(hard),
    '--label-data-root', str(labels), '--bottle-data-root', str(bottles),
    '--manual-root', str(manual),
    '--easyocr-model-dir', str(easyocr_weights), '--ocr-cache-every', '25',
    '--stage2c-checkpoint', str(stage2c_checkpoint),
    '--output-dir', str(OUTPUT), '--device', 'cuda',
    '--epochs', '60', '--patience', '10', '--batch-size', '24',
    '--feature-batch-size', '48', '--top-k', '12', '--num-workers', '2',
    '--manual-repeat', '20', '--base-temperature', '0.07', '--max-residual', '2.5',
    '--pairwise-weight', '0.25', '--protection-weight', '0.75',
    '--residual-weight', '0.02', '--pairwise-margin', '0.5',
    '--maximum-hard-val-drop', '0.002',
]
print('TRAIN COMMAND |', ' '.join(command), flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary = json.loads((OUTPUT/'evaluation.json').read_text())
print('NOTEBOOK COMPLETE |', json.dumps(summary, ensure_ascii=False), flush=True)
